In [1]:
# 1. Import libraries and load the project configuration.

%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import logging
import subprocess
import sys
import unittest
import pandas as pd
import yaml

from etl_pipeline import (
    load_config,
    validate_and_map,
    run_pipeline,
    SchemaError,
    MissingInputFileError,
    ConfigError,
    sha256_file,
    read_manifest,
)

PROJECT_DIR = Path.cwd()
CONFIG_PATH = PROJECT_DIR / "config.yaml"

config, project_root = load_config(CONFIG_PATH)
INPUT_DIR = (project_root / config["input_dir"]).resolve()

print("Project directory:", PROJECT_DIR)
print("Input directory:", INPUT_DIR)
print("Configured datasets:", len(config["datasets"]))

Project directory: c:\Users\Niranjan\Documents\sample\day14_olist_etl
Input directory: C:\Users\Niranjan\Documents\sample
Configured datasets: 9


In [2]:
# 2. Read all nine Olist CSV files.

datasets = {}

for dataset_name, rules in config["datasets"].items():
    csv_path = INPUT_DIR / rules["file"]

    if not csv_path.exists():
        print("Missing file:", rules["file"])
        continue

    data = pd.read_csv(csv_path, dtype="string", keep_default_na=False)
    datasets[dataset_name] = data

    print(
        dataset_name,
        "| rows:", len(data),
        "| columns:", len(data.columns)
    )

print("Successfully loaded:", len(datasets), "datasets")

customers | rows: 99441 | columns: 5
products | rows: 32951 | columns: 9
geolocation | rows: 1000163 | columns: 5
sellers | rows: 3095 | columns: 4
order_items | rows: 112650 | columns: 7
order_payments | rows: 103886 | columns: 5
product_category_translation | rows: 71 | columns: 2
order_reviews | rows: 99224 | columns: 7
orders | rows: 99441 | columns: 8
Successfully loaded: 9 datasets


In [3]:
# 3. Apply column mappings from config.yaml.

orders = datasets["orders"]
orders_rules = config["datasets"]["orders"]

standard_orders = orders.rename(
    columns=orders_rules.get("column_mapping", {})
)

print("Source columns:")
print(list(orders.columns))

print("Standardized columns:")
print(list(standard_orders.columns))

print("First standardized record:")
print(standard_orders.head(1).to_dict(orient="records"))

Source columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Standardized columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
First standardized record:
[{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00'}]


In [7]:
# 4. Validate records and save clean data and bad_records output.

clean_orders, bad_orders = validate_and_map(
    orders,
    "orders",
    orders_rules
)

output_dir = (project_root / config["output_dir"]).resolve()
bad_records_dir = (project_root / config["bad_records_dir"]).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

clean_orders.to_csv(output_dir / "orders_clean.csv", index=False)
clean_orders.to_json(
    output_dir / "orders_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)

if not bad_orders.empty:
    bad_records_dir.mkdir(parents=True, exist_ok=True)
    bad_orders.to_csv(
        bad_records_dir / "orders_bad_records.csv",
        index=False
    )

print("Total rows:", len(orders))
print("Valid rows:", len(clean_orders))
print("Rejected rows:", len(bad_orders))
print("Clean CSV and JSON saved.")

Total rows: 99441
Valid rows: 99441
Rejected rows: 0
Clean CSV and JSON saved.


In [8]:
# 5. Demonstrate custom exceptions and graceful failure.

try:
    invalid_sample = pd.DataFrame({"wrong_column": ["A1"]})
    validate_and_map(
        invalid_sample,
        "sample_dataset",
        {"required_columns": ["order_id"], "column_mapping": {}}
    )
except SchemaError as error:
    print("Schema error handled:", error)

try:
    missing_csv = INPUT_DIR / "file_that_does_not_exist.csv"
    if not missing_csv.exists():
        raise MissingInputFileError(f"Input file not found: {missing_csv}")
except MissingInputFileError as error:
    print("Missing-file error handled:", error)

try:
    load_config(PROJECT_DIR / "missing_config.yaml")
except ConfigError as error:
    print("Configuration error handled:", error)

print("Custom exception checks completed.")

Schema error handled: sample_dataset: missing required source columns: ['order_id']
Missing-file error handled: Input file not found: C:\Users\Niranjan\Documents\sample\file_that_does_not_exist.csv
Configuration error handled: Configuration file not found: C:\Users\Niranjan\Documents\sample\day14_olist_etl\missing_config.yaml
Custom exception checks completed.


In [9]:
# 6. Configure logging to a timestamped file.

from datetime import UTC, datetime

log_dir = (project_root / config["log_dir"]).resolve()
log_dir.mkdir(parents=True, exist_ok=True)

run_stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
notebook_logger = logging.getLogger("olist_notebook")
notebook_logger.setLevel(logging.INFO)
notebook_logger.handlers.clear()

log_path = log_dir / f"notebook_{run_stamp}.log"
file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
notebook_logger.addHandler(file_handler)

notebook_logger.info("ETL notebook started")
notebook_logger.warning("Example warning message")
notebook_logger.error("Example error message")

print("Log file created:", log_path)

Log file created: C:\Users\Niranjan\Documents\sample\day14_olist_etl\logs\notebook_20261125T094153Z.log


In [10]:
# 7. Run the full ETL pipeline and verify manifest-based idempotency.

first_run = run_pipeline(CONFIG_PATH)

print("FIRST RUN")
print("Processed:", first_run["processed_count"])
print("Skipped:", first_run["skipped_count"])
print("Failed:", first_run["failed_count"])

second_run = run_pipeline(CONFIG_PATH)

print("SECOND RUN")
print("Processed:", second_run["processed_count"])
print("Skipped:", second_run["skipped_count"])
print("Failed:", second_run["failed_count"])

# If all datasets succeeded in run 1 and inputs/config did not change,
# run 2 should process zero datasets and skip the unchanged datasets.

FIRST RUN
Processed: 9
Skipped: 0
Failed: 0
SECOND RUN
Processed: 0
Skipped: 9
Failed: 0


In [11]:
# 8. Run at least five unit tests without creating a separate tests folder.

class TestOlistETL(unittest.TestCase):

    def test_column_mapping(self):
        frame = pd.DataFrame({"source_id": ["A1"]})
        rules = {
            "required_columns": ["source_id"],
            "column_mapping": {"source_id": "standard_id"},
            "id_columns": ["source_id"]
        }
        clean, bad = validate_and_map(frame, "sample", rules)
        self.assertEqual(len(clean), 1)
        self.assertTrue(bad.empty)
        self.assertIn("standard_id", clean.columns)

    def test_invalid_record_is_rejected(self):
        frame = pd.DataFrame({"customer_id": ["C1", ""], "amount": ["10", "bad"]})
        rules = {
            "required_columns": ["customer_id"],
            "column_mapping": {},
            "id_columns": ["customer_id"],
            "numeric_columns": ["amount"]
        }
        clean, bad = validate_and_map(frame, "sample", rules)
        self.assertEqual(len(clean), 1)
        self.assertEqual(len(bad), 1)
        self.assertIn("rejection_reasons", bad.columns)

    def test_missing_required_column(self):
        frame = pd.DataFrame({"wrong": ["x"]})
        rules = {"required_columns": ["order_id"], "column_mapping": {}}
        with self.assertRaises(SchemaError):
            validate_and_map(frame, "sample", rules)

    def test_manifest_default(self):
        from tempfile import TemporaryDirectory
        with TemporaryDirectory() as temp_dir:
            manifest = read_manifest(Path(temp_dir) / "manifest.json")
            self.assertEqual(manifest, {"processed": {}})

    def test_sha256_is_deterministic(self):
        from tempfile import TemporaryDirectory
        with TemporaryDirectory() as temp_dir:
            file_path = Path(temp_dir) / "sample.txt"
            file_path.write_text("olist", encoding="utf-8")
            self.assertEqual(sha256_file(file_path), sha256_file(file_path))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestOlistETL)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)

assert test_result.wasSuccessful(), "One or more unit tests failed."
print("All five unit tests passed.")

test_column_mapping (__main__.TestOlistETL.test_column_mapping) ... ok
test_invalid_record_is_rejected (__main__.TestOlistETL.test_invalid_record_is_rejected) ... ok
test_manifest_default (__main__.TestOlistETL.test_manifest_default) ... ok
test_missing_required_column (__main__.TestOlistETL.test_missing_required_column) ... ok
test_sha256_is_deterministic (__main__.TestOlistETL.test_sha256_is_deterministic) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.022s

OK


All five unit tests passed.


In [ ]:
# 9. Prepare Git commits, feature branches, and a GitHub Pull Request.

# Run these commands from the project folder in a terminal.
# Do not run git init again if this folder is already a Git repository.

git_commands = [
    "git status",
    "git add day14_olist_etl.ipynb config.yaml etl_pipeline.py requirements.txt",
    'git commit -m "Complete Olist ETL pipeline tasks"',
    "git switch -c feature/olist-etl-improvements",
    "git push -u origin feature/olist-etl-improvements",
]

for command in git_commands:
    print(command)

# After pushing, open GitHub and create a Pull Request:
# feature/olist-etl-improvements -> main
# Review the changes, then merge the Pull Request.

In [ ]:
# 10. Live reviewer challenge: add a new column through config.yaml.

# Example: reviewer adds reviewer_segment to olist_orders_dataset.csv.
new_source_column = "reviewer_segment"
new_standard_column = "customer_segment"

orders_file = INPUT_DIR / config["datasets"]["orders"]["file"]
source_columns = pd.read_csv(orders_file, nrows=0).columns.tolist()

if new_source_column in source_columns:
    with CONFIG_PATH.open("r", encoding="utf-8") as file:
        live_config = yaml.safe_load(file)

    live_config["datasets"]["orders"]["column_mapping"][new_source_column] = (
        new_standard_column
    )

    with CONFIG_PATH.open("w", encoding="utf-8") as file:
        yaml.safe_dump(live_config, file, sort_keys=False, allow_unicode=True)

    print("Mapping added:", new_source_column, "->", new_standard_column)

    live_result = run_pipeline(CONFIG_PATH)
    print("Live pipeline result:", live_result)

    updated_orders = pd.read_csv(
        output_dir / "orders" / "orders_clean.csv",
        dtype="string"
    )
    print("New standard column present:",
          new_standard_column in updated_orders.columns)
else:
    print(
        f"{new_source_column} is not in the CSV yet. "
        "Ask the reviewer to add it, then rerun this cell."
    )